In [1]:
# Cell 1：基础环境

from pathlib import Path
import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RESULTS_DIR = PROJECT_ROOT / "results"

In [2]:
# Cell 2：加载三模型测试预测

svm_df = pd.read_csv(
    RESULTS_DIR / "svm_test_predictions.csv"
)

textcnn_df = pd.read_csv(
    RESULTS_DIR / "textcnn_test_predictions.csv"
)

distilbert_df = pd.read_csv(
    RESULTS_DIR / "distilbert_test_predictions.csv"
)

print("SVM:", svm_df.shape)
print("TextCNN:", textcnn_df.shape)
print("DistilBERT:", distilbert_df.shape)

SVM: (1961, 4)
TextCNN: (1961, 4)
DistilBERT: (1961, 5)


In [3]:
# Cell 3：确认同一批测试样本完全对齐

assert svm_df["text"].reset_index(drop=True).equals(
    textcnn_df["text"].reset_index(drop=True)
)

assert svm_df["text"].reset_index(drop=True).equals(
    distilbert_df["text"].reset_index(drop=True)
)

assert svm_df["category"].reset_index(drop=True).equals(
    textcnn_df["category"].reset_index(drop=True)
)

assert svm_df["category"].reset_index(drop=True).equals(
    distilbert_df["category"].reset_index(drop=True)
)

print("All three model predictions are aligned.")

All three model predictions are aligned.


In [4]:
# Cell 4：统一模型比较DataFrame

model_comparison_df = pd.DataFrame({
    "text": svm_df["text"],
    "true_category": svm_df["category"],

    "svm_prediction": svm_df["svm_prediction"],

    "textcnn_prediction":
        textcnn_df["textcnn_prediction"],

    "distilbert_prediction":
        distilbert_df["distilbert_prediction"]
})

for model_name in [
    "svm",
    "textcnn",
    "distilbert"
]:
    model_comparison_df[
        f"{model_name}_correct"
    ] = (
        model_comparison_df[
            f"{model_name}_prediction"
        ]
        ==
        model_comparison_df[
            "true_category"
        ]
    )

display(model_comparison_df.head())

,text,true_category,svm_prediction,textcnn_prediction,distilbert_prediction,svm_correct,textcnn_correct,distilbert_correct
0,Will my transfer immediately show up in my acc...,transfer_timing,pending_transfer,transfer_timing,balance_not_updated_after_bank_transfer,False,True,False
1,I don't have my access code for the app.,passcode_forgotten,passcode_forgotten,passcode_forgotten,passcode_forgotten,True,True,True
2,How long do bank transfers take to clear,balance_not_updated_after_bank_transfer,balance_not_updated_after_bank_transfer,balance_not_updated_after_bank_transfer,balance_not_updated_after_bank_transfer,True,True,True
3,Are there any fees if I want to add money usin...,top_up_by_card_charge,top_up_by_card_charge,top_up_by_card_charge,top_up_by_card_charge,True,True,True
4,How low does the balance have to be before the...,automatic_top_up,automatic_top_up,balance_not_updated_after_bank_transfer,transfer_timing,True,False,False


In [5]:
# Cell 5：DistilBERT vs SVM

svm_fixed_by_distilbert = (
    model_comparison_df[
        (~model_comparison_df["svm_correct"])
        &
        (model_comparison_df["distilbert_correct"])
    ]
    .copy()
)

distilbert_new_errors_vs_svm = (
    model_comparison_df[
        (model_comparison_df["svm_correct"])
        &
        (~model_comparison_df["distilbert_correct"])
    ]
    .copy()
)

net_gain_vs_svm = (
    len(svm_fixed_by_distilbert)
    -
    len(distilbert_new_errors_vs_svm)
)

print(
    "SVM errors fixed by DistilBERT:",
    len(svm_fixed_by_distilbert)
)

print(
    "New DistilBERT errors vs SVM:",
    len(distilbert_new_errors_vs_svm)
)

print(
    "Net gain vs SVM:",
    net_gain_vs_svm
)

SVM errors fixed by DistilBERT: 111
New DistilBERT errors vs SVM: 35
Net gain vs SVM: 76


In [6]:
# Cell 6：DistilBERT vs TextCNN

textcnn_fixed_by_distilbert = (
    model_comparison_df[
        (~model_comparison_df["textcnn_correct"])
        &
        (model_comparison_df["distilbert_correct"])
    ]
    .copy()
)

distilbert_new_errors_vs_textcnn = (
    model_comparison_df[
        (model_comparison_df["textcnn_correct"])
        &
        (~model_comparison_df["distilbert_correct"])
    ]
    .copy()
)

net_gain_vs_textcnn = (
    len(textcnn_fixed_by_distilbert)
    -
    len(distilbert_new_errors_vs_textcnn)
)

print(
    "TextCNN errors fixed by DistilBERT:",
    len(textcnn_fixed_by_distilbert)
)

print(
    "New DistilBERT errors vs TextCNN:",
    len(distilbert_new_errors_vs_textcnn)
)

print(
    "Net gain vs TextCNN:",
    net_gain_vs_textcnn
)

TextCNN errors fixed by DistilBERT: 122
New DistilBERT errors vs TextCNN: 40
Net gain vs TextCNN: 82


In [7]:
# Cell 7：三个模型同时正确/同时错误

all_correct_df = model_comparison_df[
    model_comparison_df["svm_correct"]
    &
    model_comparison_df["textcnn_correct"]
    &
    model_comparison_df["distilbert_correct"]
].copy()

all_wrong_df = model_comparison_df[
    (~model_comparison_df["svm_correct"])
    &
    (~model_comparison_df["textcnn_correct"])
    &
    (~model_comparison_df["distilbert_correct"])
].copy()

print(
    "All three correct:",
    len(all_correct_df)
)

print(
    "All three wrong:",
    len(all_wrong_df)
)

All three correct: 1627
All three wrong: 95


In [8]:
# Cell 8：只有DistilBERT预测正确的样本

distilbert_only_correct_df = (
    model_comparison_df[
        (~model_comparison_df["svm_correct"])
        &
        (~model_comparison_df["textcnn_correct"])
        &
        (model_comparison_df["distilbert_correct"])
    ]
    .copy()
)

print(
    "Only DistilBERT correct:",
    len(distilbert_only_correct_df)
)

display(
    distilbert_only_correct_df[
        [
            "text",
            "true_category",
            "svm_prediction",
            "textcnn_prediction",
            "distilbert_prediction"
        ]
    ].head(20)
)

Only DistilBERT correct: 48


,text,true_category,svm_prediction,textcnn_prediction,distilbert_prediction
141,What's the process for terminating an account?,terminate_account,age_limit,age_limit,terminate_account
185,I tried to send someone money but they haven't...,transfer_not_received_by_recipient,reverted_card_payment?,wrong_amount_of_cash_received,transfer_not_received_by_recipient
190,How does one send me money?,receiving_money,transfer_into_account,transfer_into_account,receiving_money
237,help me with my transfer,failed_transfer,declined_transfer,declined_transfer,failed_transfer
238,"I want to top up by transfering accounts, What...",transfer_into_account,topping_up_by_card,top_up_failed,transfer_into_account
240,When will the transfer go through?,pending_transfer,failed_transfer,transfer_not_received_by_recipient,pending_transfer
266,What currencies can be used for top up?,supported_cards_and_currencies,fiat_currency_support,transfer_into_account,supported_cards_and_currencies
288,What stipulations do disposable cards entail?,disposable_card_limits,get_disposable_virtual_card,get_disposable_virtual_card,disposable_card_limits
289,When will my transfer go through?,pending_transfer,failed_transfer,transfer_not_received_by_recipient,pending_transfer
310,I don't think the charges made when I had curr...,card_payment_wrong_exchange_rate,wrong_exchange_rate_for_cash_withdrawal,wrong_exchange_rate_for_cash_withdrawal,card_payment_wrong_exchange_rate


In [9]:
# Cell 9：统计三模型正确/错误组合

prediction_pattern_df = (
    model_comparison_df
    .groupby(
        [
            "svm_correct",
            "textcnn_correct",
            "distilbert_correct"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        by="count",
        ascending=False
    )
)

display(
    prediction_pattern_df
)

,svm_correct,textcnn_correct,distilbert_correct,count
7,True,True,True,1627
0,False,False,False,95
5,True,False,True,74
3,False,True,True,63
1,False,False,True,48
6,True,True,False,21
2,False,True,False,19
4,True,False,False,14


In [10]:
# Cell 10：只有DistilBERT正确的样本

distilbert_only_correct_df = (
    model_comparison_df[
        (~model_comparison_df["svm_correct"])
        &
        (~model_comparison_df["textcnn_correct"])
        &
        (model_comparison_df["distilbert_correct"])
    ]
    .copy()
)

print(
    "Only DistilBERT correct:",
    len(distilbert_only_correct_df)
)

display(
    distilbert_only_correct_df[
        [
            "text",
            "true_category",
            "svm_prediction",
            "textcnn_prediction",
            "distilbert_prediction"
        ]
    ]
)

Only DistilBERT correct: 48


,text,true_category,svm_prediction,textcnn_prediction,distilbert_prediction
141,What's the process for terminating an account?,terminate_account,age_limit,age_limit,terminate_account
185,I tried to send someone money but they haven't...,transfer_not_received_by_recipient,reverted_card_payment?,wrong_amount_of_cash_received,transfer_not_received_by_recipient
190,How does one send me money?,receiving_money,transfer_into_account,transfer_into_account,receiving_money
237,help me with my transfer,failed_transfer,declined_transfer,declined_transfer,failed_transfer
238,"I want to top up by transfering accounts, What...",transfer_into_account,topping_up_by_card,top_up_failed,transfer_into_account
240,When will the transfer go through?,pending_transfer,failed_transfer,transfer_not_received_by_recipient,pending_transfer
266,What currencies can be used for top up?,supported_cards_and_currencies,fiat_currency_support,transfer_into_account,supported_cards_and_currencies
288,What stipulations do disposable cards entail?,disposable_card_limits,get_disposable_virtual_card,get_disposable_virtual_card,disposable_card_limits
289,When will my transfer go through?,pending_transfer,failed_transfer,transfer_not_received_by_recipient,pending_transfer
310,I don't think the charges made when I had curr...,card_payment_wrong_exchange_rate,wrong_exchange_rate_for_cash_withdrawal,wrong_exchange_rate_for_cash_withdrawal,card_payment_wrong_exchange_rate


In [11]:
# Cell 11：DistilBERT独有修复集中在哪些类别

distilbert_unique_fixes_by_category = (
    distilbert_only_correct_df
    .groupby(
        "true_category"
    )
    .size()
    .reset_index(
        name="fixed_count"
    )
    .sort_values(
        by="fixed_count",
        ascending=False
    )
)

display(
    distilbert_unique_fixes_by_category
    .head(15)
)

,true_category,fixed_count
19,request_refund,5
17,pending_transfer,4
3,card_payment_wrong_exchange_rate,2
2,card_not_working,2
11,fiat_currency_support,2
23,top_up_by_bank_transfer_charge,2
7,disposable_card_limits,2
4,change_pin,2
24,top_up_failed,2
28,transfer_not_received_by_recipient,2


In [12]:
# Cell 12：查看DistilBERT独有修复的文本

for _, row in (
    distilbert_only_correct_df
    .head(25)
    .iterrows()
):

    print("=" * 90)

    print(
        "TEXT:",
        row["text"]
    )

    print(
        "TRUE:",
        row["true_category"]
    )

    print(
        "SVM:",
        row["svm_prediction"]
    )

    print(
        "TextCNN:",
        row["textcnn_prediction"]
    )

    print(
        "DistilBERT:",
        row["distilbert_prediction"]
    )

TEXT: What's the process for terminating an account?
TRUE: terminate_account
SVM: age_limit
TextCNN: age_limit
DistilBERT: terminate_account
TEXT: I tried to send someone money but they haven't received it.
TRUE: transfer_not_received_by_recipient
SVM: reverted_card_payment?
TextCNN: wrong_amount_of_cash_received
DistilBERT: transfer_not_received_by_recipient
TEXT: How does one send me money?
TRUE: receiving_money
SVM: transfer_into_account
TextCNN: transfer_into_account
DistilBERT: receiving_money
TEXT: help me with my transfer
TRUE: failed_transfer
SVM: declined_transfer
TextCNN: declined_transfer
DistilBERT: failed_transfer
TEXT: I want to top up by transfering accounts, What do I do?
TRUE: transfer_into_account
SVM: topping_up_by_card
TextCNN: top_up_failed
DistilBERT: transfer_into_account
TEXT: When will the transfer go through?
TRUE: pending_transfer
SVM: failed_transfer
TextCNN: transfer_not_received_by_recipient
DistilBERT: pending_transfer
TEXT: What currencies can be used fo

In [13]:
# Cell 13：三个模型共同错误

all_wrong_df = (
    model_comparison_df[
        (~model_comparison_df["svm_correct"])
        &
        (~model_comparison_df["textcnn_correct"])
        &
        (~model_comparison_df["distilbert_correct"])
    ]
    .copy()
)

print(
    "All three wrong:",
    len(all_wrong_df)
)

display(
    all_wrong_df[
        [
            "text",
            "true_category",
            "svm_prediction",
            "textcnn_prediction",
            "distilbert_prediction"
        ]
    ]
    .head(30)
)

All three wrong: 95


,text,true_category,svm_prediction,textcnn_prediction,distilbert_prediction
69,HOW LONG TO TAKE THE TIME TO SOLVE,card_not_working,card_delivery_estimate,transfer_not_received_by_recipient,card_delivery_estimate
117,what is the leadtime for a new card to show up...,card_delivery_estimate,card_arrival,card_arrival,card_arrival
156,As far as courtries go which ones are supported?,country_support,wrong_amount_of_cash_received,failed_transfer,supported_cards_and_currencies
244,What cards do you offer?,visa_or_mastercard,getting_spare_card,supported_cards_and_currencies,supported_cards_and_currencies
301,my transfer isn't there,balance_not_updated_after_bank_transfer,transfer_not_received_by_recipient,transfer_not_received_by_recipient,transfer_not_received_by_recipient
321,Are there any hindrances to using a disposable...,disposable_card_limits,get_disposable_virtual_card,get_disposable_virtual_card,get_disposable_virtual_card
339,Someone else charged my card!,direct_debit_payment_not_recognised,compromised_card,card_payment_fee_charged,card_payment_fee_charged
377,I think someone stole money from my card,cash_withdrawal_not_recognised,lost_or_stolen_card,compromised_card,compromised_card
388,What is not accepted in order for me to top up...,top_up_by_cash_or_cheque,top_up_by_bank_transfer_charge,topping_up_by_card,topping_up_by_card
401,Do you accept Mastercard?,supported_cards_and_currencies,visa_or_mastercard,visa_or_mastercard,visa_or_mastercard


In [14]:
# Cell 14：共同错误类别统计

shared_error_categories_df = (
    all_wrong_df
    .groupby(
        "true_category"
    )
    .size()
    .reset_index(
        name="error_count"
    )
    .sort_values(
        by="error_count",
        ascending=False
    )
)

display(
    shared_error_categories_df
    .head(15)
)

,true_category,error_count
2,balance_not_updated_after_bank_transfer,5
14,direct_debit_payment_not_recognised,5
25,pending_top_up,5
13,declined_transfer,4
6,card_payment_not_recognised,4
31,top_up_failed,4
26,pending_transfer,4
5,card_not_working,4
30,top_up_by_cash_or_cheque,3
28,supported_cards_and_currencies,3
